# Eidomira — measure the neural path on a free Kaggle GPU

**Before running anything, two settings in the right-hand panel:**

| setting | value | why |
|---|---|---|
| Accelerator | **GPU T4 x2** | without it every number below is a CPU number |
| Internet | **On** | needed to clone the repository and install packages |

Then **Run All**. It takes about three minutes, most of which is `pip`.

**What this answers:** does the neural path run on a real GPU, and how fast — mask, blend,
restorer, one swap pass, and the four-pass pixel boost, turned into frames per second at the
three widths the adaptive controller targets.

**What it does not answer:** quality. The licensed swap weights are not bought yet, so it
measures shape-correct stand-ins (the plumbing, and the host) rather than a real swap. Point
`--models` at a directory of real weights later and it measures those instead.


In [ ]:
# Two settings in the right sidebar: Accelerator = GPU T4 x2, Internet = On.
BRANCH = "arena/178efbef-eidomira"

!nvidia-smi -L
!git clone -q -b {BRANCH} https://github.com/Phillipjr9/eidomira.git
%cd eidomira
!git log --oneline -1


In [ ]:
# A GPU build of ONNX Runtime, plus insightface for the swap model.
# The uninstall first is deliberate: Kaggle ships a CPU-only onnxruntime, and both
# distributions install into the same directory — leaving the CPU one behind is how a
# notebook ends up quietly measuring a processor.
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu insightface onnx

# insightface asks for GUI opencv, which needs libGL. Kaggle usually has it; if it does not,
# cv2 fails to import and the headless build fixes it. Reinstalling only on failure keeps the
# common case fast.
!python -c "import cv2" || python -m pip install -q --force-reinstall opencv-python-headless

# The one line that decides whether the numbers below mean anything.
!python -c "import onnxruntime as o; print(o.get_available_providers())"


In [ ]:
!python tools/gpu_validation.py


## Reading the output

`will be used` is the provider the **application** will pick, resolved by `app/providers.py` —
the same code the studio runs, not a guess. If it says `CPUExecutionProvider`, nothing below it
is a GPU measurement, and the script says so rather than letting the number flatter itself.

The frame figures are an extrapolation: cost is scaled by pixel count from the 1280×720 frame
measured, which is honest for a convolution-shaped model and a rough guide for a real one. The
pixel boost runs four swap passes, so add roughly three times the swap time at scale=2.

**Save the output** (or screenshot it) — it is the number that decides which GPU is worth
renting, and it is worth more than any specification table.

A notebook cannot host the live studio: no public IP, no inbound UDP, no persistent process.
This is for measurement, not serving. See `docs/elastic-compute-plan.md`.
